# Introduction

# Albotherm AXF — Exploratory Data Analysis
### Pipeline Step 5 of 5: Clean → Merge → EDA → Testing → **Model**

Experiments requiring laboratory testing will be designed using the full factorial model.

After the experiments are completed, RSM will be used to find the most ideal optimal region and Bayesion Optimization will be used to identify the most ideal optimal conditions

# Import necessary library

In [9]:
import itertools

# Design factorial experiments

Define the 3 levels for each factor <br>
These levels are centered around the current promising region

In [10]:
frr_levels = [6.0, 7.0, 8.0]
dfr_levels = [24.0, 27.0, 30.0]
imp_levels = [50.0, 81.0, 112.0]

center_point = {
    "Flow Rate Ratio": 7.0,
    "Dispersed Flow Rate (mL/min)": 27.0,
    "Impellor Speed (rpm)": 112.0
}

print("Flow Rate Ratio levels:", frr_levels)
print("Dispersed Flow Rate levels:", dfr_levels)
print("Impellor Speed levels:", imp_levels)
print("Center point:", center_point)

Flow Rate Ratio levels: [6.0, 7.0, 8.0]
Dispersed Flow Rate levels: [24.0, 27.0, 30.0]
Impellor Speed levels: [50.0, 81.0, 112.0]
Center point: {'Flow Rate Ratio': 7.0, 'Dispersed Flow Rate (mL/min)': 27.0, 'Impellor Speed (rpm)': 112.0}


Generate the full 3 x 3 x 3 factorial design (27 runs)

In [11]:
design_27 = pd.DataFrame(
    list(itertools.product(frr_levels, dfr_levels, imp_levels)),
    columns=[
        "Flow Rate Ratio",
        "Dispersed Flow Rate (mL/min)",
        "Impellor Speed (rpm)"
    ]
)

print("Number of factorial runs:", len(design_27))
design_27.head()

Number of factorial runs: 27


,Flow Rate Ratio,Dispersed Flow Rate (mL/min),Impellor Speed (rpm)
0,6.0,24.0,50.0
1,6.0,24.0,81.0
2,6.0,24.0,112.0
3,6.0,27.0,50.0
4,6.0,27.0,81.0


Create 3 replicated center-point runs

In [12]:
center_replicates = pd.DataFrame(
    [center_point] * 3
)

print("Number of center replicates:", len(center_replicates))
center_replicates

Number of center replicates: 3


,Flow Rate Ratio,Dispersed Flow Rate (mL/min),Impellor Speed (rpm)
0,7.0,27.0,112.0
1,7.0,27.0,112.0
2,7.0,27.0,112.0


Combine factorial runs and center replicates to make 30 runs total

In [13]:
design_30 = pd.concat([design_27, center_replicates], ignore_index=True)

print("Total number of runs:", len(design_30))
design_30.tail()

Total number of runs: 30


,Flow Rate Ratio,Dispersed Flow Rate (mL/min),Impellor Speed (rpm)
25,8.0,30.0,81.0
26,8.0,30.0,112.0
27,7.0,27.0,112.0
28,7.0,27.0,112.0
29,7.0,27.0,112.0


In [14]:
design_30["Run_Type"] = ["Factorial"] * len(design_27) + ["Center_Replicate"] * len(center_replicates)
design_30.insert(0, "Run", range(1, len(design_30) + 1))

design_30

,Run,Flow Rate Ratio,Dispersed Flow Rate (mL/min),Impellor Speed (rpm),Run_Type
0,1,6.0,24.0,50.0,Factorial
1,2,6.0,24.0,81.0,Factorial
2,3,6.0,24.0,112.0,Factorial
3,4,6.0,27.0,50.0,Factorial
4,5,6.0,27.0,81.0,Factorial
5,6,6.0,27.0,112.0,Factorial
6,7,6.0,30.0,50.0,Factorial
7,8,6.0,30.0,81.0,Factorial
8,9,6.0,30.0,112.0,Factorial
9,10,7.0,24.0,50.0,Factorial


Caculate code value for RSM optimization

In [15]:
design_30_coded = design_30.copy()

design_30_coded["Flow Rate Ratio_coded"] = (
    design_30_coded["Flow Rate Ratio"] - frr_levels[1]
) / ((frr_levels[2] - frr_levels[0]) / 2)

design_30_coded["Dispersed Flow Rate (mL/min)_coded"] = (
    design_30_coded["Dispersed Flow Rate (mL/min)"] - dfr_levels[1]
) / ((dfr_levels[2] - dfr_levels[0]) / 2)

design_30_coded["Impellor Speed (rpm)_coded"] = (
    design_30_coded["Impellor Speed (rpm)"] - imp_levels[1]
) / ((imp_levels[2] - imp_levels[0]) / 2)

design_30_coded

,Run,Flow Rate Ratio,Dispersed Flow Rate (mL/min),Impellor Speed (rpm),Run_Type,Flow Rate Ratio_coded,Dispersed Flow Rate (mL/min)_coded,Impellor Speed (rpm)_coded
0,1,6.0,24.0,50.0,Factorial,-1.0,-1.0,-1.0
1,2,6.0,24.0,81.0,Factorial,-1.0,-1.0,0.0
2,3,6.0,24.0,112.0,Factorial,-1.0,-1.0,1.0
3,4,6.0,27.0,50.0,Factorial,-1.0,0.0,-1.0
4,5,6.0,27.0,81.0,Factorial,-1.0,0.0,0.0
5,6,6.0,27.0,112.0,Factorial,-1.0,0.0,1.0
6,7,6.0,30.0,50.0,Factorial,-1.0,1.0,-1.0
7,8,6.0,30.0,81.0,Factorial,-1.0,1.0,0.0
8,9,6.0,30.0,112.0,Factorial,-1.0,1.0,1.0
9,10,7.0,24.0,50.0,Factorial,0.0,-1.0,-1.0


Randomize the run order for actual experimentation -> reduce bias from time/order effects

In [16]:
design_30_randomized = design_30_coded.sample(frac=1, random_state=42).reset_index(drop=True)
design_30_randomized["Run"] = range(1, len(design_30_randomized) + 1)

design_30_randomized

,Run,Flow Rate Ratio,Dispersed Flow Rate (mL/min),Impellor Speed (rpm),Run_Type,Flow Rate Ratio_coded,Dispersed Flow Rate (mL/min)_coded,Impellor Speed (rpm)_coded
0,1,7.0,27.0,112.0,Center_Replicate,0.0,0.0,1.0
1,2,7.0,30.0,50.0,Factorial,0.0,1.0,-1.0
2,3,8.0,27.0,112.0,Factorial,1.0,0.0,1.0
3,4,7.0,30.0,112.0,Factorial,0.0,1.0,1.0
4,5,6.0,30.0,112.0,Factorial,-1.0,1.0,1.0
5,6,7.0,24.0,50.0,Factorial,0.0,-1.0,-1.0
6,7,7.0,27.0,112.0,Center_Replicate,0.0,0.0,1.0
7,8,8.0,30.0,50.0,Factorial,1.0,1.0,-1.0
8,9,7.0,27.0,50.0,Factorial,0.0,0.0,-1.0
9,10,6.0,24.0,50.0,Factorial,-1.0,-1.0,-1.0


These experiments will be sent to a laboratory to collect the best-performing results for determining the ideal conditions to optimize the desired outcome.